# Usine A100 80 Go : distiller Bonsai 2 27B et entrainer le clone de Jev, puis exporter pour la RTX 4060

Ce notebook execute les phases 6 (distillation), 7 (entrainement RLCD-lite) et l'export GGUF du
[protocole](https://github.com/speed25200-cyber/LLM-and-Classifier/blob/claude/jev-bonsai-fusion-rtx4060-ish8th/docs/00-PROTOCOLE-FUSION-JEV-BONSAI.md)
sur une **A100 80 Go** (Colab Pro/Pro+ : Runtime > Change runtime type > A100). Le produit final (GGUF Q8_0 du clone
+ `calibration.json`) se deploie ensuite sur la 4060 avec `JEV_GGUF=... ./scripts/start_jev_clone.sh`.

Duree indicative : installation 10 min, distillation ~0,4 s/etat (Bonsai 2 PQ2_0, 4 slots), entrainement 2B complet
~4-6 h pour 180 M tokens. Sauvegarder sur Google Drive : une session Colab est ephemere (12-24 h max).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv   # attendu : NVIDIA A100-SXM4-80GB
import torch, os; print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.get_device_name(0))
from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/jev-bonsai'; os.makedirs(DRIVE, exist_ok=True)

In [ ]:
%cd /content
!git clone -q -b claude/jev-bonsai-fusion-rtx4060-ish8th https://github.com/speed25200-cyber/LLM-and-Classifier.git
%cd /content/LLM-and-Classifier
!pip install -q -e ".[serve,dev,train]" huggingface_hub datasets
# binaires llama.cpp du fork PrismML (CUDA detecte) + Bonsai 2 27B PQ2_0 + mmproj + Ternary-Bonsai-8B (clone niveau 0)
!PROFILE=scripts/profiles/a100-80gb.env SKIP_VENV=1 sh scripts/setup.sh

Si les binaires precompiles refusent de demarrer (pilote CUDA trop ancien), compiler depuis les sources (~10 min sur l'A100) :
```
!git clone -q -b prism https://github.com/PrismML-Eng/llama.cpp && cd llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=80 -DCMAKE_BUILD_TYPE=Release >/dev/null && cmake --build build -j8 --target llama-server llama-quantize llama-bench >/dev/null && mkdir -p ../bin/cuda && cp build/bin/* ../bin/cuda/
```

In [ ]:
# --- System Two : Bonsai 2 27B PQ2_0 en arriere-plan (port 8080) ---
import subprocess, time, requests, os
env = dict(os.environ, PROFILE='scripts/profiles/a100-80gb.env')
bonsai = subprocess.Popen(['sh', 'scripts/start_bonsai.sh'], env=env, stdout=open('runs_bonsai.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(180):
    time.sleep(2)
    try:
        if requests.get('http://127.0.0.1:8080/health', timeout=2).json().get('status') == 'ok': break
    except Exception: pass
print(requests.get('http://127.0.0.1:8080/health').json())
r = requests.post('http://127.0.0.1:8080/v1/chat/completions', json={'messages': [{'role': 'user', 'content': 'Implement binary search in Python.'}], 'max_tokens': 200, 'thinking_budget_tokens': 256}).json()
print(r['timings']['predicted_per_second'], 'tok/s generation ;', r['timings']['prompt_per_second'], 'tok/s prefill')

## Donnees
1. **Publiques** (generalisation) : `training/make_public_mix.py` (banking77, ag_news, go_emotions, MMLU, yelp).
2. **Vos etats** (domaine) : un JSONL `{"state": ..., "questions": {...}}` depose dans Drive (`states.jsonl`).
   Bonsai les etiquette (mode `soft` = lecture System One sur Bonsai, escalade `think` sous 0,85 de confiance).

In [ ]:
!python training/make_public_mix.py --out data/public_train.jsonl --val data/public_val.jsonl --per-task 8000
import os, shutil
own = f'{DRIVE}/states.jsonl'
if os.path.exists(own):
    !python -m jev_clone.distill --bonsai http://127.0.0.1:8080 --in $own --out data/labeled.jsonl --mode soft --think-if-below 0.85 --budget 2048
    !shuf data/labeled.jsonl > data/labeled_shuf.jsonl && head -n -300 data/labeled_shuf.jsonl > data/domain_train.jsonl && tail -n 300 data/labeled_shuf.jsonl > data/domain_val.jsonl
    !cat data/public_train.jsonl data/domain_train.jsonl data/domain_train.jsonl > data/train.jsonl   # domaine x2
    !cat data/public_val.jsonl data/domain_val.jsonl > data/val.jsonl
else:
    print('pas de states.jsonl dans Drive : entrainement sur donnees publiques seulement')
    !cp data/public_train.jsonl data/train.jsonl && cp data/public_val.jsonl data/val.jsonl
!wc -l data/train.jsonl data/val.jsonl

## Entrainement RLCD-lite (fine-tuning complet, recette decider)
`Qwen/Qwen3.5-2B-Base` : le clone final pese ~2,1 Go en Q8_0 (profil 4060 « vitesse », a cote de Bonsai-27B 1-bit) ou ~1,3 Go en Q4_K_M.
Pour le profil 4060 « qualite » (Bonsai 2 + clone), preferer `Qwen/Qwen3.5-0.8B-Base` (~0,9 Go en Q8_0).
Objectif = NLL sur les logits restreints aux etiquettes + KL vers les distributions de Bonsai quand elles existent.

In [ ]:
MODEL = 'Qwen/Qwen3.5-2B-Base'     # ou 'Qwen/Qwen3.5-0.8B-Base' pour le profil qualite 8 Go
!python training/train_lora_rlcd.py --model $MODEL --full --data data/train.jsonl --val data/val.jsonl \
    --out runs/jev-clone --epochs 1 --bs 16 --accum 2 --max-len 1536 --loss nll --kl 0.5 --permutations 2 --save-every 500 2>&1 | tail -40
!cp -r runs/jev-clone $DRIVE/ 2>/dev/null; print('copie dans Drive')

## Export GGUF (pour `scripts/start_jev_clone.sh` sur la 4060) et calibration finale sur llama-server

In [ ]:
!git clone -q -b prism --depth 1 https://github.com/PrismML-Eng/llama.cpp /content/llama.cpp-src
!pip install -q -r /content/llama.cpp-src/requirements/requirements-convert_hf_to_gguf.txt 2>/dev/null || pip install -q gguf sentencepiece
!python /content/llama.cpp-src/convert_hf_to_gguf.py runs/jev-clone/merged --outfile runs/jev-clone-f16.gguf --outtype f16
BIN = 'bin/cuda'
!LD_LIBRARY_PATH=$BIN $BIN/llama-quantize runs/jev-clone-f16.gguf runs/jev-clone-Q8_0.gguf Q8_0 | tail -2
!LD_LIBRARY_PATH=$BIN $BIN/llama-quantize runs/jev-clone-f16.gguf runs/jev-clone-Q4_K_M.gguf Q4_K_M | tail -2
!ls -la runs/*.gguf

In [ ]:
# clone servi par llama-server (port 8081) : calibration (temperature + seuils) et latence
clone = subprocess.Popen(['sh', 'scripts/start_jev_clone.sh'], env=dict(env, JEV_GGUF='runs/jev-clone-Q8_0.gguf'),
                         stdout=open('runs_clone.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(90):
    time.sleep(2)
    try:
        if requests.get('http://127.0.0.1:8081/health', timeout=2).json().get('status') == 'ok': break
    except Exception: pass
!python -m jev_clone.calibrate --server http://127.0.0.1:8081 --data data/val.jsonl --out runs/calibration.json --target-precision 0.95
!jev bench --server http://127.0.0.1:8081 --n 30

In [ ]:
# artefacts a rapatrier sur la 4060 : GGUF + calibration (+ merged HF pour re-entrainer plus tard)
!cp runs/jev-clone-Q8_0.gguf runs/jev-clone-Q4_K_M.gguf runs/calibration.json $DRIVE/
!ls -la $DRIVE
print('''Sur la 4060 :
  JEV_GGUF=models/jev-clone-Q8_0.gguf ./scripts/start_jev_clone.sh
  JEV_CALIBRATION=runs/calibration.json JEV_S1_URL=http://127.0.0.1:8081 JEV_S2_URL=http://127.0.0.1:8080 jev serve''')

## Option : evaluer toute la fusion sur l'A100
Les deux serveurs tournent deja : `JEV_S1_URL=http://127.0.0.1:8081 JEV_S2_URL=http://127.0.0.1:8080 python examples/ticket_routing.py`
mesure le taux d'escalade et les latences ; `runs/ledger.jsonl` alimente la prochaine iteration (phase 8).